# TechStore — First Week Report

Initial SQL report on TechStore's products, stock levels, customer base, orders, and sales targets.  
The sample database is loaded locally into DuckDB from `SQL/seed_techstore.sql`.

In [1]:
from pathlib import Path
import duckdb

con = duckdb.connect()
con.execute(Path("SQL/seed_techstore.sql").read_text(encoding="utf-8"))


def sql(query: str):
    """Run a SQL query against the in-memory TechStore database and return a DataFrame."""
    return con.sql(query).df()


sql("SHOW TABLES")

,name
0,clientes
1,itens_pedido
2,pedidos
3,produtos
4,vendedores


## 1. Product Catalog

List all products ordered by category (`categoria`) and price (`preco`, highest first). Show `nome`, `categoria`, `preco`, and `estoque`.

In [2]:
query = """
SELECT nome, categoria, preco, estoque
FROM produtos
ORDER BY categoria ASC, preco DESC;
"""
sql(query)

,nome,categoria,preco,estoque
0,Teclado Mecanico Pro,Acessorios,349.9,150
1,Mouse Gamer RGB,Acessorios,189.9,300
2,Adaptador USB-C Hub,Acessorios,149.9,200
3,Suporte Notebook Aluminio,Acessorios,129.9,85
4,Cabo HDMI 2.1,Acessorios,59.9,95
5,Mousepad Gamer XL,Acessorios,49.9,110
6,"Pelicula Tablet 10""",Acessorios,39.9,150
7,"Capa Notebook 15""",Acessorios,29.9,120
8,"Monitor UltraWide 34""",Monitores,2899.0,5
9,"Monitor Gamer 27"" 144Hz",Monitores,1799.0,22


**Note:** `Notebooks` has the highest-priced items (up to R$ 7,899.00 for the `Notebook Gamer Z5`), while `Acessorios` has the widest variety (8 SKUs) and the highest inventory counts (85–300 units per item).

## 2. Low Stock Items

Find products with fewer than 15 units in stock (`estoque < 15`), ordered from lowest to highest stock. Show `nome`, `categoria`, `estoque`, and `preco`.

In [3]:
query = """
SELECT nome, categoria, estoque, preco
FROM produtos
WHERE estoque < 15
ORDER BY estoque ASC;
"""
sql(query)

,nome,categoria,estoque,preco
0,"Monitor UltraWide 34""",Monitores,5,2899.0
1,Notebook Gamer Z5,Notebooks,8,7899.0
2,Workstation W3,Notebooks,12,2599.0


**Note:** Only 3 products fall below 15 units (`Monitor UltraWide 34"` with 5, `Notebook Gamer Z5` with 8, and `Workstation W3` with 12), but all three are high-ticket items (R$ 2,599–R$ 7,899). Running out of stock on these hits revenue much harder than running low on accessories, so they should be first in line for restocking.

## 3. Customer Base by State

List all customers ordered by state (`estado`) and name (`nome`). Show `nome`, `cidade`, `estado`, and `segmento`.

In [4]:
query = """
SELECT nome, cidade, estado, segmento
FROM clientes
ORDER BY estado ASC, nome ASC;
"""
sql(query)

,nome,cidade,estado,segmento
0,Prefeitura Regional AM,Manaus,AM,governo
1,Roberto Dias,Manaus,AM,varejo
2,ComTech BA,Salvador,BA,corporativo
3,Roberto Alves,Salvador,BA,varejo
4,Secretaria de Educacao BA,Salvador,BA,governo
5,Beatriz Nogueira,Fortaleza,CE,varejo
6,StartUp XYZ,Brasilia,DF,corporativo
7,Carla Mendonca,Goiania,GO,varejo
8,Ana Oliveira,Belo Horizonte,MG,varejo
9,Camila Santos,Juiz de Fora,MG,varejo


**Note:** Most of the 47 customers are concentrated in the Southeast (`SP`, `RJ`, `MG`), with smaller pockets in the South (`PR`, `RS`, `SC`) and Northeast (`PE`, `BA`, `CE`), and very few accounts in the North and Center-West.

## 4. Government Segment Customers

List customers in the `governo` segment, ordered alphabetically by `nome`. Show `nome`, `cidade`, `estado`, and `segmento`.

In [5]:
query = """
SELECT nome, cidade, estado, segmento
FROM clientes
WHERE segmento = 'governo'
ORDER BY nome ASC;
"""
sql(query)

,nome,cidade,estado,segmento
0,Governo MG Edu,Belo Horizonte,MG,governo
1,Governo RJ Digital,Rio de Janeiro,RJ,governo
2,Prefeitura Regional AM,Manaus,AM,governo
3,Prefeitura de Recife,Recife,PE,governo
4,Recife Digital,Recife,PE,governo
5,Secretaria de Educacao BA,Salvador,BA,governo
6,TechGov SP,Sao Paulo,SP,governo


**Note:** There are 7 government accounts spread across 6 states (`AM`, `BA`, `MG`, `PE`, `RJ`, `SP`). Because public-sector procurement usually involves tenders and longer approval cycles, tracking these separately from retail accounts makes sense for forecasting.

## 5. Top 10 Orders by Value

List the 10 largest orders by `valor_total`. Show `id`, `data_pedido`, `valor_total`, `status`, and `forma_pagamento`.

In [6]:
query = """
SELECT id, data_pedido, valor_total, status, forma_pagamento
FROM pedidos
ORDER BY valor_total DESC
LIMIT 10;
"""
sql(query)

,id,data_pedido,valor_total,status,forma_pagamento
0,34,2024-02-20,12450.0,entregue,cartao
1,67,2024-03-05,8799.9,entregue,pix
2,91,2024-03-11,8799.0,pendente,cartao
3,4,2023-12-04,7899.0,entregue,cartao
4,12,2024-01-18,7899.0,entregue,cartao
5,44,2024-02-09,7899.0,entregue,cartao
6,52,2024-02-20,7899.0,entregue,cartao
7,94,2024-03-12,7899.0,entregue,cartao
8,155,2024-03-09,6480.0,pendente,cartao
9,41,2024-02-02,5480.0,cancelado,cartao


**Note:** The largest order is `#34` (R$ 12,450.00, paid by card, delivered). Worth flagging: two of the top 10 orders (`#91` at R$ 8,799.00 and `#155` at R$ 6,480.00) are still `pendente`, and `#41` (R$ 5,480.00) was `cancelado`.

## 6. Cancelled Orders

List all orders with `status = 'cancelado'`, ordered from highest to lowest `valor_total`. Show `id`, `data_pedido`, `valor_total`, and `forma_pagamento`.

In [7]:
query = """
SELECT id, data_pedido, valor_total, forma_pagamento
FROM pedidos
WHERE status = 'cancelado'
ORDER BY valor_total DESC;
"""
sql(query)

,id,data_pedido,valor_total,forma_pagamento
0,41,2024-02-02,5480.0,cartao
1,132,2024-01-17,4599.0,cartao
2,136,2024-03-10,4599.0,cartao
3,15,2024-01-10,3299.0,boleto
4,133,2024-01-20,2583.1,pix
5,73,2024-03-08,2199.0,pix
6,29,2024-01-25,1899.0,cartao
7,137,2024-03-12,1899.0,cartao
8,138,2024-03-14,1163.0,boleto
9,23,2024-01-15,899.9,boleto


**Note:** There are 12 cancelled orders totaling R$ 28,940.00. Card payments (`cartao`) account for half of them (6 orders, R$ 15,280.00), including the three largest cancellations (`#41`, `#136`, `#132`). Checking whether those are payment-gateway declines or buyer drop-offs would be a good follow-up.

## 7. Sales Reps Above R$ 50,000 Target

List sales reps with a monthly target (`meta_mensal`) above R$ 50,000, ordered from highest to lowest target. Show `nome`, `regiao`, and `meta_mensal`.

In [8]:
query = """
SELECT nome, regiao, meta_mensal
FROM vendedores
WHERE meta_mensal > 50000
ORDER BY meta_mensal DESC;
"""
sql(query)

,nome,regiao,meta_mensal
0,Ricardo Mendes,Sudeste,60000.0
1,Carlos Eduardo,Sudeste,55000.0


**Note:** Only 2 of the 7 sales reps have targets above R$ 50,000 — `Ricardo Mendes` (R$ 60,000) and `Carlos Eduardo` (R$ 55,000). Both cover `Sudeste`, which lines up with where most of the customer base is located.